# 🖥️ SWASH tutorial

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import parcels
import parcels.tutorial

In [ ]:
swash_files = parcels.tutorial.open_dataset("SWASH_data/data", download_only=True)

In [ ]:
ds = parcels.convert.swash_to_sgrid(
    data_file=swash_files[0], coord_file=swash_files[1], total_depth=8.0
)
fieldset = parcels.FieldSet.from_sgrid_conventions(ds)
fieldset.describe()

In [ ]:
npart = 10  # number of particles to be released
y = np.linspace(1, 16, npart)
x = np.repeat(15, npart)

layerI = 0  # at which water depth, the particles are released
z = np.repeat(ds.depth.values[layerI], npart)
pset = parcels.ParticleSet(fieldset, x=x, y=y, z=z)

output_file = parcels.ParticleFile(
    "output-swash.parquet", outputdt=np.timedelta64(5, "s"), mode="w"
)
pset.execute(
    parcels.kernels.AdvectionRK2,
    runtime=np.timedelta64(20, "s"),
    dt=np.timedelta64(1, "s"),
    output_file=output_file,
    verbose_progress=False,
)

In [ ]:
df = parcels.read_particlefile("output-swash.parquet")

fig, ax = plt.subplots(figsize=(8, 4))
waterlevel = ds.isel(time=2).watlev.plot(cmap="magma", ax=ax)
for traj in df.partition_by("particle_id"):
    ax.plot(traj["x"][0], traj["y"][0], "wo", markersize=5)
    ax.plot(traj["x"], traj["y"], color="k")
ax.set_xlim([14, 16])
plt.show()